
# Election Bloc Change Prediction Project  
## Notebook 01 — Data Loading and Party-to-Bloc Mapping

### Purpose

This notebook creates a clean and audited election-results table for the 21st through 25th Knesset elections.

The notebook deliberately focuses only on election data. Demographic, socioeconomic and locality-level data will be processed in later notebooks.

### Main tasks

1. Locate and audit the raw project files.
2. Load the five election-result files with robust Hebrew-encoding handling.
3. Map parties and ballot letters into four modeled blocs:
   - `Right`
   - `Center_Left`
   - `Haredi`
   - `Arab`
4. Retain all unmapped votes under `Other`.
5. Preserve turnout-related variables for later analysis.
6. Run collision, coverage, duplication and composition checks.
7. Save a reusable election-level dataset and mapping-audit reports.

### Main outputs

- `data/interim/election_bloc_results.csv`
- `reports/tables/party_mapping_long.csv`
- `reports/tables/election_mapping_audit.csv`
- `reports/tables/election_coverage_summary.csv`
- `reports/summaries/notebook_01_summary.json`


## 1. Imports and project paths

In [ ]:

from pathlib import Path
import json
import os
import platform
import re
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 100)

RANDOM_STATE = 99


In [ ]:

def locate_repo_root():
    """Locate the repository root without depending on one Colab path."""
    candidates = []

    explicit_root = os.getenv("ELECTION_PROJECT_ROOT")
    if explicit_root:
        candidates.append(Path(explicit_root).expanduser())

    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])

    candidates.extend([
        Path("/content/Election_Bloc_Prediction_Project"),
        Path("/content/Election_Bloc_Change_Project"),
    ])

    checked = set()

    for candidate in candidates:
        candidate = candidate.resolve()

        if candidate in checked:
            continue

        checked.add(candidate)

        if (candidate / "data" / "raw").exists():
            return candidate

    checked_text = "\n".join(f"- {path}" for path in checked)

    raise FileNotFoundError(
        "Could not locate a repository containing data/raw. "
        "Run the notebook from the repository or set ELECTION_PROJECT_ROOT.\n"
        f"Checked:\n{checked_text}"
    )


REPO_ROOT = locate_repo_root()

RAW_DIR = REPO_ROOT / "data" / "raw"
ELECTION_DIR = RAW_DIR / "Election_data"

INTERIM_DIR = REPO_ROOT / "data" / "interim"
REPORTS_DIR = REPO_ROOT / "reports"
TABLES_DIR = REPORTS_DIR / "tables"
SUMMARIES_DIR = REPORTS_DIR / "summaries"
FIGURES_DIR = REPORTS_DIR / "figures"

for directory in [
    INTERIM_DIR,
    TABLES_DIR,
    SUMMARIES_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Election directory:", ELECTION_DIR)
print("Interim output directory:", INTERIM_DIR)



## 2. Expected raw-data inventory

This stage checks that the five election files are available. It also records whether the 2019–2023 demographic, education, income and welfare files needed by later notebooks are present.

Only missing election files stop this notebook. Missing future feature files are reported for follow-up.


In [ ]:

ELECTION_FILES = {
    "Knesset_21": "Knesset_21_Election_Results_2019_By_Locality.csv",
    "Knesset_22": "Knesset_22_Election_Results_2019_By_Locality.csv",
    "Knesset_23": "Knesset_23_Election_Results_2020_By_Locality.csv",
    "Knesset_24": "Knesset_24_Election_Results_2021_By_Locality.csv",
    "Knesset_25": "Knesset_25_Election_Results_2022_By_Locality.csv",
}

ELECTION_METADATA = {
    "Knesset_21": {"election_number": 21, "year": 2019},
    "Knesset_22": {"election_number": 22, "year": 2019},
    "Knesset_23": {"election_number": 23, "year": 2020},
    "Knesset_24": {"election_number": 24, "year": 2021},
    "Knesset_25": {"election_number": 25, "year": 2022},
}

expected_sources = []

for election_name, filename in ELECTION_FILES.items():
    expected_sources.append({
        "source_group": "Election_data",
        "year_or_election": election_name,
        "path": ELECTION_DIR / filename,
        "required_now": True,
    })

for year in range(2019, 2024):
    expected_sources.extend([
        {
            "source_group": "Education_data",
            "year_or_election": year,
            "path": RAW_DIR / "Education_data" / f"Education_{year}.xlsx",
            "required_now": False,
        },
        {
            "source_group": "Demographic_data",
            "year_or_election": year,
            "path": RAW_DIR / "Demographic_data" / f"Demographic_{year}.xlsx",
            "required_now": False,
        },
        {
            "source_group": "Average_income",
            "year_or_election": year,
            "path": RAW_DIR / "Average_income" / f"ICBS_{year}.xlsx",
            "required_now": False,
        },
        {
            "source_group": "Unemployment_data",
            "year_or_election": year,
            "path": RAW_DIR / "Unemployment_data" / f"Unemployment_{year}.xlsx",
            "required_now": False,
        },
    ])

expected_sources.append({
    "source_group": "Locality_Types",
    "year_or_election": "static",
    "path": RAW_DIR / "Locality_Types.xlsx",
    "required_now": False,
})

source_inventory = pd.DataFrame(expected_sources)
source_inventory["exists"] = source_inventory["path"].map(Path.exists)
source_inventory["path"] = source_inventory["path"].astype(str)

missing_required = source_inventory.query("required_now and not exists")

if not missing_required.empty:
    raise FileNotFoundError(
        "Required election files are missing:\n"
        + "\n".join(missing_required["path"].tolist())
    )

source_inventory.groupby(
    ["source_group", "required_now"],
    as_index=False,
)["exists"].agg(["count", "sum"]).reset_index()



## 3. Party-to-bloc definitions

The main definition is coalitional rather than purely ideological:

- `Right` approximates the Netanyahu bloc, excluding the Haredi parties.
- `Center_Left` approximates the anti-Netanyahu / change bloc for the period.
- Yisrael Beiteinu is therefore assigned to `Center_Left` in the primary analysis.
- Haredi and Arab parties are modeled as separate blocs.

The mapping is defined by election because ballot letters can be reused by different lists in different elections. This prevents a letter used in one election from being incorrectly assigned in another election.


In [ ]:

MODELED_BLOCS = [
    "Right",
    "Center_Left",
    "Haredi",
    "Arab",
]

YISRAEL_BEITEINU_BLOC = "Center_Left"

PARTY_MAPPING_ROWS = [
    # Knesset 21
    ("Knesset_21", "מחל", "הליכוד", "Right"),
    ("Knesset_21", "טב", "איחוד מפלגות הימין", "Right"),
    ("Knesset_21", "כ", "כולנו", "Right"),
    ("Knesset_21", "נ", "הימין החדש", "Right"),
    ("Knesset_21", "ז", "זהות", "Right"),
    ("Knesset_21", "פה", "כחול לבן", "Center_Left"),
    ("Knesset_21", "אמת", "העבודה", "Center_Left"),
    ("Knesset_21", "מרצ", "מרצ", "Center_Left"),
    ("Knesset_21", "נר", "גשר", "Center_Left"),
    ("Knesset_21", "ל", "ישראל ביתנו", YISRAEL_BEITEINU_BLOC),
    ("Knesset_21", "שס", 'ש"ס', "Haredi"),
    ("Knesset_21", "ג", "יהדות התורה", "Haredi"),
    ("Knesset_21", "ום", 'חד"ש-תע"ל', "Arab"),
    ("Knesset_21", "דעם", 'רע"ם-בל"ד', "Arab"),

    # Knesset 22
    ("Knesset_22", "מחל", "הליכוד", "Right"),
    ("Knesset_22", "טב", "ימינה", "Right"),
    ("Knesset_22", "כף", "עוצמה יהודית", "Right"),
    ("Knesset_22", "פה", "כחול לבן", "Center_Left"),
    ("Knesset_22", "אמת", "העבודה-גשר", "Center_Left"),
    ("Knesset_22", "מרצ", "המחנה הדמוקרטי", "Center_Left"),
    ("Knesset_22", "ל", "ישראל ביתנו", YISRAEL_BEITEINU_BLOC),
    ("Knesset_22", "שס", 'ש"ס', "Haredi"),
    ("Knesset_22", "ג", "יהדות התורה", "Haredi"),
    ("Knesset_22", "ודעם", "הרשימה המשותפת", "Arab"),

    # Knesset 23
    ("Knesset_23", "מחל", "הליכוד", "Right"),
    ("Knesset_23", "טב", "ימינה", "Right"),
    ("Knesset_23", "נץ", "עוצמה יהודית", "Right"),
    ("Knesset_23", "פה", "כחול לבן", "Center_Left"),
    ("Knesset_23", "אמת", "העבודה-גשר-מרצ", "Center_Left"),
    ("Knesset_23", "ל", "ישראל ביתנו", YISRAEL_BEITEINU_BLOC),
    ("Knesset_23", "שס", 'ש"ס', "Haredi"),
    ("Knesset_23", "ג", "יהדות התורה", "Haredi"),
    ("Knesset_23", "ודעם", "הרשימה המשותפת", "Arab"),

    # Knesset 24
    ("Knesset_24", "מחל", "הליכוד", "Right"),
    ("Knesset_24", "ב", "ימינה", "Right"),
    ("Knesset_24", "ט", "הציונות הדתית", "Right"),
    ("Knesset_24", "ת", "תקווה חדשה", "Right"),
    ("Knesset_24", "פה", "יש עתיד", "Center_Left"),
    ("Knesset_24", "אמת", "העבודה", "Center_Left"),
    ("Knesset_24", "מרצ", "מרצ", "Center_Left"),
    ("Knesset_24", "כן", "כחול לבן", "Center_Left"),
    ("Knesset_24", "ל", "ישראל ביתנו", YISRAEL_BEITEINU_BLOC),
    ("Knesset_24", "שס", 'ש"ס', "Haredi"),
    ("Knesset_24", "ג", "יהדות התורה", "Haredi"),
    ("Knesset_24", "ודעם", "הרשימה המשותפת", "Arab"),
    ("Knesset_24", "עם", 'רע"ם', "Arab"),

    # Knesset 25
    ("Knesset_25", "מחל", "הליכוד", "Right"),
    ("Knesset_25", "ב", "הבית היהודי", "Right"),
    ("Knesset_25", "ט", "הציונות הדתית ועוצמה יהודית", "Right"),
    ("Knesset_25", "פה", "יש עתיד", "Center_Left"),
    ("Knesset_25", "אמת", "העבודה", "Center_Left"),
    ("Knesset_25", "מרצ", "מרצ", "Center_Left"),
    ("Knesset_25", "כן", "המחנה הממלכתי", "Center_Left"),
    ("Knesset_25", "ל", "ישראל ביתנו", YISRAEL_BEITEINU_BLOC),
    ("Knesset_25", "שס", 'ש"ס', "Haredi"),
    ("Knesset_25", "ג", "יהדות התורה", "Haredi"),
    ("Knesset_25", "עם", 'רע"ם', "Arab"),
    ("Knesset_25", "ום", 'חד"ש-תע"ל', "Arab"),
    ("Knesset_25", "ד", 'בל"ד', "Arab"),
]

party_mapping = pd.DataFrame(
    PARTY_MAPPING_ROWS,
    columns=[
        "target_election",
        "ballot_letter",
        "party_name",
        "bloc",
    ],
)

party_mapping["election_number"] = party_mapping[
    "target_election"
].map(
    {
        election: metadata["election_number"]
        for election, metadata in ELECTION_METADATA.items()
    }
)

party_mapping = party_mapping.sort_values(
    ["election_number", "bloc", "ballot_letter"]
).reset_index(drop=True)

party_mapping.head(20)


## 4. Mapping validation and collision checks

In [ ]:

invalid_blocs = sorted(
    set(party_mapping["bloc"]) - set(MODELED_BLOCS)
)

if invalid_blocs:
    raise ValueError(f"Invalid bloc names: {invalid_blocs}")

duplicate_mapping = party_mapping[
    party_mapping.duplicated(
        ["target_election", "ballot_letter"],
        keep=False,
    )
]

if not duplicate_mapping.empty:
    raise ValueError(
        "The same ballot letter is mapped more than once "
        "within the same election:\n"
        + duplicate_mapping.to_string(index=False)
    )

configured_elections = set(ELECTION_FILES)
mapped_elections = set(party_mapping["target_election"])

if configured_elections != mapped_elections:
    raise ValueError(
        "Election mismatch between file configuration and mapping. "
        f"Configured: {sorted(configured_elections)}; "
        f"mapped: {sorted(mapped_elections)}"
    )

print("Mapping validation passed.")
print("Mapped election-specific ballot rows:", len(party_mapping))
print("Primary Yisrael Beiteinu bloc:", YISRAEL_BEITEINU_BLOC)


## 5. Robust election-file loading

In [ ]:

def clean_column_name(column):
    return str(column).replace("\ufeff", "").strip()


def read_election_csv(file_path):
    """Read an election CSV using common Hebrew encodings."""
    encoding_attempts = [
        "utf-8-sig",
        "utf-8",
        "cp1255",
    ]

    errors = []

    for encoding in encoding_attempts:
        try:
            df = pd.read_csv(file_path, encoding=encoding)
            df.columns = [clean_column_name(col) for col in df.columns]
            return df, encoding
        except UnicodeDecodeError as error:
            errors.append(f"{encoding}: {error}")

    raise UnicodeError(
        f"Could not decode {file_path}. Attempts:\n"
        + "\n".join(errors)
    )


def first_existing_column(df, candidates, required=True):
    for candidate in candidates:
        if candidate in df.columns:
            return candidate

    if required:
        raise KeyError(
            f"None of the expected columns were found: {candidates}"
        )

    return None


def numeric_series(series):
    """Convert vote/count columns to numeric safely."""
    cleaned = (
        series.astype("string")
        .str.replace(",", "", regex=False)
        .str.strip()
    )

    return pd.to_numeric(cleaned, errors="coerce")


def normalize_locality_symbol(series):
    numeric = numeric_series(series).astype("Int64")
    return numeric.astype("string")


NON_PARTY_COLUMNS = {
    "סמל ועדה",
    "שם ישוב",
    "שם יישוב",
    "שם היישוב",
    "סמל ישוב",
    "סמל יישוב",
    "סמל היישוב",
    "בזב",
    "מצביעים",
    "פסולים",
    "כשרים",
    "ריכוז",
    "שופט",
    "ברזל",
}


def identify_party_columns(df):
    party_columns = []

    for column in df.columns:
        if column in NON_PARTY_COLUMNS:
            continue

        if column.lower().startswith("unnamed"):
            continue

        numeric_values = numeric_series(df[column])

        if numeric_values.notna().any():
            party_columns.append(column)

    return party_columns


In [ ]:

raw_elections = {}
encoding_rows = []

for election_name, filename in ELECTION_FILES.items():
    file_path = ELECTION_DIR / filename
    raw_df, encoding = read_election_csv(file_path)

    raw_elections[election_name] = raw_df

    encoding_rows.append({
        "target_election": election_name,
        "file_name": filename,
        "encoding_used": encoding,
        "rows": len(raw_df),
        "columns": raw_df.shape[1],
    })

encoding_audit = pd.DataFrame(encoding_rows)
encoding_audit



## 6. Ballot-level mapping audit

Every numeric ballot column is audited. Unmapped lists are retained under `Other`, rather than silently discarded.

The notebook stops if a single unmapped ballot letter exceeds 1% of valid votes in an election.


In [ ]:

UNMAPPED_ALERT_THRESHOLD_PCT = 1.0
MANUAL_REVIEW_THRESHOLD_PCT = 0.05

mapping_lookup = {
    election_name: (
        election_mapping.set_index("ballot_letter")
        [["party_name", "bloc"]]
        .to_dict(orient="index")
    )
    for election_name, election_mapping
    in party_mapping.groupby("target_election")
}


def build_mapping_audit(raw_elections):
    audit_rows = []

    for election_name, raw_df in raw_elections.items():
        valid_column = first_existing_column(
            raw_df,
            ["כשרים"],
        )

        valid_total = numeric_series(
            raw_df[valid_column]
        ).sum()

        election_mapping = mapping_lookup[election_name]
        party_columns = identify_party_columns(raw_df)

        for ballot_letter in party_columns:
            votes = numeric_series(
                raw_df[ballot_letter]
            ).fillna(0).sum()

            share_pct = (
                votes / valid_total * 100
                if valid_total > 0
                else np.nan
            )

            mapping_info = election_mapping.get(ballot_letter)

            audit_rows.append({
                "target_election": election_name,
                "election_number": ELECTION_METADATA[
                    election_name
                ]["election_number"],
                "year": ELECTION_METADATA[election_name]["year"],
                "ballot_letter": ballot_letter,
                "mapped": mapping_info is not None,
                "party_name": (
                    mapping_info["party_name"]
                    if mapping_info
                    else pd.NA
                ),
                "bloc": (
                    mapping_info["bloc"]
                    if mapping_info
                    else "Other"
                ),
                "votes": int(votes),
                "share_of_valid_votes_pct": share_pct,
            })

    return pd.DataFrame(audit_rows).sort_values(
        [
            "election_number",
            "mapped",
            "share_of_valid_votes_pct",
        ],
        ascending=[True, True, False],
    ).reset_index(drop=True)


mapping_audit = build_mapping_audit(raw_elections)

missing_expected_mapping = party_mapping.merge(
    mapping_audit[
        ["target_election", "ballot_letter"]
    ].drop_duplicates(),
    on=["target_election", "ballot_letter"],
    how="left",
    indicator=True,
).query("_merge == 'left_only'")

if not missing_expected_mapping.empty:
    raise ValueError(
        "Mapped ballot letters are missing from their configured "
        "election files:\n"
        + missing_expected_mapping[
            [
                "target_election",
                "ballot_letter",
                "party_name",
            ]
        ].to_string(index=False)
    )

large_unmapped = mapping_audit.query(
    "not mapped and "
    "share_of_valid_votes_pct > "
    "@UNMAPPED_ALERT_THRESHOLD_PCT"
)

if not large_unmapped.empty:
    raise ValueError(
        "At least one unmapped ballot letter exceeds the "
        f"{UNMAPPED_ALERT_THRESHOLD_PCT}% alert threshold:\n"
        + large_unmapped.to_string(index=False)
    )

manual_review_unmapped = mapping_audit.query(
    "not mapped and "
    "share_of_valid_votes_pct >= "
    "@MANUAL_REVIEW_THRESHOLD_PCT"
)

manual_review_unmapped[
    [
        "target_election",
        "ballot_letter",
        "votes",
        "share_of_valid_votes_pct",
    ]
].reset_index(drop=True)


## 7. Aggregate parties into the four modeled blocs

In [ ]:

def optional_numeric_column(df, candidates, default=0):
    column = first_existing_column(
        df,
        candidates,
        required=False,
    )

    if column is None:
        return pd.Series(
            default,
            index=df.index,
            dtype="float64",
        )

    return numeric_series(df[column]).fillna(default)


def aggregate_election(raw_df, election_name):
    df = raw_df.copy()

    symbol_column = first_existing_column(
        df,
        ["סמל ישוב", "סמל יישוב", "סמל היישוב"],
    )

    name_column = first_existing_column(
        df,
        ["שם ישוב", "שם יישוב", "שם היישוב"],
        required=False,
    )

    valid_column = first_existing_column(
        df,
        ["כשרים"],
    )

    valid_votes = numeric_series(df[valid_column])

    if valid_votes.isna().any():
        raise ValueError(
            f"{election_name}: missing or invalid valid-vote values."
        )

    locality_symbol = normalize_locality_symbol(
        df[symbol_column]
    )

    missing_symbol = locality_symbol.isna()

    if missing_symbol.any():
        print(
            f"Warning: {election_name} contains "
            f"{int(missing_symbol.sum())} rows without a valid "
            "locality symbol. These rows will be removed."
        )

    election_mapping = mapping_lookup[election_name]

    bloc_counts = pd.DataFrame(
        0.0,
        index=df.index,
        columns=MODELED_BLOCS,
    )

    for ballot_letter, mapping_info in election_mapping.items():
        votes = numeric_series(
            df[ballot_letter]
        ).fillna(0)

        bloc_counts[
            mapping_info["bloc"]
        ] += votes

    total_main_blocs = bloc_counts.sum(axis=1)
    other_votes = valid_votes - total_main_blocs

    materially_negative_other = other_votes < -0.5

    if materially_negative_other.any():
        raise ValueError(
            f"{election_name}: mapped votes exceed valid votes "
            f"in {int(materially_negative_other.sum())} rows."
        )

    other_votes = other_votes.clip(lower=0)

    eligible_voters = optional_numeric_column(
        df,
        ["בזב"],
        default=np.nan,
    )

    votes_cast = optional_numeric_column(
        df,
        ["מצביעים"],
        default=np.nan,
    )

    invalid_votes = optional_numeric_column(
        df,
        ["פסולים"],
        default=np.nan,
    )

    output = pd.DataFrame({
        "locality_symbol": locality_symbol,
        "election_locality_name": (
            df[name_column].astype("string").str.strip()
            if name_column is not None
            else pd.Series(pd.NA, index=df.index, dtype="string")
        ),
        "target_election": election_name,
        "election_number": ELECTION_METADATA[
            election_name
        ]["election_number"],
        "year": ELECTION_METADATA[election_name]["year"],
        "eligible_voters": eligible_voters,
        "votes_cast": votes_cast,
        "invalid_votes": invalid_votes,
        "valid_votes": valid_votes,
    })

    for bloc in MODELED_BLOCS:
        output[bloc] = bloc_counts[bloc]

    output["Other"] = other_votes
    output["total_main_blocs"] = total_main_blocs

    output["turnout_pct"] = np.where(
        output["eligible_voters"] > 0,
        output["votes_cast"]
        / output["eligible_voters"]
        * 100,
        np.nan,
    )

    output["invalid_vote_pct"] = np.where(
        output["votes_cast"] > 0,
        output["invalid_votes"]
        / output["votes_cast"]
        * 100,
        np.nan,
    )

    output["modeled_vote_coverage_pct"] = np.where(
        output["valid_votes"] > 0,
        output["total_main_blocs"]
        / output["valid_votes"]
        * 100,
        np.nan,
    )

    output["Other_raw_pct"] = np.where(
        output["valid_votes"] > 0,
        output["Other"]
        / output["valid_votes"]
        * 100,
        np.nan,
    )

    for bloc in MODELED_BLOCS:
        output[f"{bloc}_valid_pct"] = np.where(
            output["valid_votes"] > 0,
            output[bloc]
            / output["valid_votes"]
            * 100,
            np.nan,
        )

        output[f"{bloc}_pct"] = np.where(
            output["total_main_blocs"] > 0,
            output[bloc]
            / output["total_main_blocs"]
            * 100,
            np.nan,
        )

    output = output.loc[
        ~missing_symbol
    ].copy()

    return output


processed_elections = {
    election_name: aggregate_election(
        raw_df,
        election_name,
    )
    for election_name, raw_df in raw_elections.items()
}

election_bloc_results = pd.concat(
    processed_elections.values(),
    ignore_index=True,
)

election_bloc_results = election_bloc_results.sort_values(
    ["election_number", "locality_symbol"],
    key=lambda series: (
        pd.to_numeric(series, errors="coerce")
        if series.name == "locality_symbol"
        else series
    ),
).reset_index(drop=True)

election_bloc_results.head()


## 8. Coverage summary

In [ ]:

coverage_rows = []

for election_name, group in election_bloc_results.groupby(
    "target_election",
    sort=False,
):
    valid_votes = group["valid_votes"].sum()
    modeled_votes = group["total_main_blocs"].sum()

    coverage_rows.append({
        "target_election": election_name,
        "election_number": int(
            group["election_number"].iloc[0]
        ),
        "year": int(group["year"].iloc[0]),
        "locality_rows": len(group),
        "unique_localities": group[
            "locality_symbol"
        ].nunique(),
        "eligible_voters": group[
            "eligible_voters"
        ].sum(min_count=1),
        "votes_cast": group[
            "votes_cast"
        ].sum(min_count=1),
        "valid_votes": valid_votes,
        "modeled_votes": modeled_votes,
        "other_votes": group["Other"].sum(),
        "modeled_vote_coverage_pct": (
            modeled_votes / valid_votes * 100
        ),
        "other_pct_of_valid_votes": (
            group["Other"].sum()
            / valid_votes
            * 100
        ),
        "national_turnout_pct": (
            group["votes_cast"].sum()
            / group["eligible_voters"].sum()
            * 100
        ),
    })

coverage_summary = pd.DataFrame(
    coverage_rows
).sort_values("election_number")

coverage_summary


In [ ]:

ax = coverage_summary.plot(
    x="target_election",
    y="modeled_vote_coverage_pct",
    kind="bar",
    legend=False,
    figsize=(9, 4),
)

ax.set_title(
    "Share of valid votes covered by the four modeled blocs"
)
ax.set_xlabel("Election")
ax.set_ylabel("Coverage (%)")
ax.set_ylim(
    max(
        0,
        coverage_summary[
            "modeled_vote_coverage_pct"
        ].min() - 1
    ),
    100,
)
ax.tick_params(axis="x", rotation=0)

plt.tight_layout()

coverage_figure_path = (
    FIGURES_DIR
    / "election_mapping_coverage.png"
)

plt.savefig(
    coverage_figure_path,
    dpi=150,
    bbox_inches="tight",
)

plt.show()

print("Figure saved to:", coverage_figure_path)


## 9. Data-quality checks

In [ ]:

TARGET_COLUMNS = [
    f"{bloc}_pct"
    for bloc in MODELED_BLOCS
]

quality_checks = {}

duplicate_count = int(
    election_bloc_results.duplicated(
        ["locality_symbol", "target_election"]
    ).sum()
)

quality_checks[
    "duplicate_locality_election_rows"
] = duplicate_count

if duplicate_count:
    raise ValueError(
        f"Found {duplicate_count} duplicated "
        "locality-election rows."
    )

negative_count_columns = [
    *MODELED_BLOCS,
    "Other",
    "total_main_blocs",
    "valid_votes",
]

negative_counts = {
    column: int(
        (election_bloc_results[column] < 0).sum()
    )
    for column in negative_count_columns
}

quality_checks[
    "negative_value_counts"
] = negative_counts

if any(negative_counts.values()):
    raise ValueError(
        f"Negative vote/count values found: {negative_counts}"
    )

modeled_rows = (
    election_bloc_results[
        "total_main_blocs"
    ] > 0
)

target_sums = election_bloc_results.loc[
    modeled_rows,
    TARGET_COLUMNS,
].sum(axis=1)

max_composition_error = float(
    (target_sums - 100).abs().max()
)

quality_checks[
    "max_target_sum_absolute_error"
] = max_composition_error

if max_composition_error > 1e-8:
    raise ValueError(
        "The normalized bloc targets do not sum to 100. "
        f"Maximum error: {max_composition_error}"
    )

reconstructed_valid_votes = (
    election_bloc_results[
        MODELED_BLOCS
    ].sum(axis=1)
    + election_bloc_results["Other"]
)

max_vote_reconstruction_error = float(
    (
        reconstructed_valid_votes
        - election_bloc_results["valid_votes"]
    ).abs().max()
)

quality_checks[
    "max_valid_vote_reconstruction_error"
] = max_vote_reconstruction_error

if max_vote_reconstruction_error > 0.5:
    raise ValueError(
        "Mapped bloc counts plus Other do not reconstruct "
        "valid votes."
    )

largest_unmapped_share = float(
    mapping_audit.loc[
        ~mapping_audit["mapped"],
        "share_of_valid_votes_pct",
    ].max()
)

quality_checks[
    "largest_single_unmapped_ballot_share_pct"
] = largest_unmapped_share

print("All quality checks passed.")
pd.Series(quality_checks, dtype="object")



## 10. Save outputs

The election-level output retains both:

- Shares relative to all valid votes, for audit and coverage analysis.
- Four-bloc normalized targets that sum to 100%, for later compositional modeling.

`locality_symbol` and locality names are identifiers only. They will not be used directly as predictive features.


In [ ]:

ELECTION_OUTPUT_PATH = (
    INTERIM_DIR
    / "election_bloc_results.csv"
)

PARTY_MAPPING_PATH = (
    TABLES_DIR
    / "party_mapping_long.csv"
)

MAPPING_AUDIT_PATH = (
    TABLES_DIR
    / "election_mapping_audit.csv"
)

COVERAGE_SUMMARY_PATH = (
    TABLES_DIR
    / "election_coverage_summary.csv"
)

SOURCE_INVENTORY_PATH = (
    TABLES_DIR
    / "raw_source_inventory.csv"
)

NOTEBOOK_SUMMARY_PATH = (
    SUMMARIES_DIR
    / "notebook_01_summary.json"
)

election_bloc_results.to_csv(
    ELECTION_OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

party_mapping.to_csv(
    PARTY_MAPPING_PATH,
    index=False,
    encoding="utf-8-sig",
)

mapping_audit.to_csv(
    MAPPING_AUDIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

coverage_summary.to_csv(
    COVERAGE_SUMMARY_PATH,
    index=False,
    encoding="utf-8-sig",
)

source_inventory.to_csv(
    SOURCE_INVENTORY_PATH,
    index=False,
    encoding="utf-8-sig",
)

summary = {
    "notebook": "01_data_loading_and_party_mapping",
    "elections_processed": len(ELECTION_FILES),
    "rows_created": int(len(election_bloc_results)),
    "unique_localities_across_all_elections": int(
        election_bloc_results[
            "locality_symbol"
        ].nunique()
    ),
    "mapped_election_ballot_rows": int(
        len(party_mapping)
    ),
    "main_blocs": MODELED_BLOCS,
    "yisrael_beiteinu_primary_bloc": (
        YISRAEL_BEITEINU_BLOC
    ),
    "unmapped_alert_threshold_pct": (
        UNMAPPED_ALERT_THRESHOLD_PCT
    ),
    "largest_single_unmapped_ballot_share_pct": (
        largest_unmapped_share
    ),
    "max_target_sum_absolute_error": (
        max_composition_error
    ),
    "max_valid_vote_reconstruction_error": (
        max_vote_reconstruction_error
    ),
    "outputs": {
        "election_bloc_results": str(
            ELECTION_OUTPUT_PATH.relative_to(
                REPO_ROOT
            )
        ),
        "party_mapping": str(
            PARTY_MAPPING_PATH.relative_to(
                REPO_ROOT
            )
        ),
        "mapping_audit": str(
            MAPPING_AUDIT_PATH.relative_to(
                REPO_ROOT
            )
        ),
        "coverage_summary": str(
            COVERAGE_SUMMARY_PATH.relative_to(
                REPO_ROOT
            )
        ),
        "source_inventory": str(
            SOURCE_INVENTORY_PATH.relative_to(
                REPO_ROOT
            )
        ),
    },
}

with NOTEBOOK_SUMMARY_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

print("Saved:")
for path in [
    ELECTION_OUTPUT_PATH,
    PARTY_MAPPING_PATH,
    MAPPING_AUDIT_PATH,
    COVERAGE_SUMMARY_PATH,
    SOURCE_INVENTORY_PATH,
    NOTEBOOK_SUMMARY_PATH,
]:
    print("-", path)


## 11. Final output preview

In [ ]:

preview_columns = [
    "locality_symbol",
    "election_locality_name",
    "target_election",
    "year",
    "eligible_voters",
    "votes_cast",
    "valid_votes",
    "turnout_pct",
    "Right_pct",
    "Center_Left_pct",
    "Haredi_pct",
    "Arab_pct",
    "Other_raw_pct",
]

election_bloc_results[
    preview_columns
].head(10)



## Notebook 01 completion checklist

This notebook is complete when:

- All five election files load successfully.
- The election-specific party mapping passes collision checks.
- No single unmapped ballot letter exceeds the alert threshold.
- The four normalized bloc shares sum to 100%.
- Bloc counts plus `Other` reconstruct valid votes.
- No locality-election duplicates remain.
- Turnout and invalid-vote variables are retained.
- All audit and output files are saved.

The next notebook is:

> `02_panel_and_target_engineering.ipynb`

It will transform the election-level data into consecutive election transitions and create previous-election, percentage-point delta and CLR-delta targets.
